In [1]:
using LowLevelFEM, LinearAlgebra
using IterativeSolvers, Preconditioners

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 2 for different dependency version already loaded, 7 for different Julia build configuration, 1 for file size changed)
Precompiling packages...
   9.9 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 11 seconds. 134 already precompiled.
[ Info: Precompiling LowLevelFEMIterativeSolversExt [55978d5f-b2ed-541f-a526-69ba3ceeb2d1] (caches not reused: 1 for different dependency version already loaded, 4 for different Julia build configuration)
Precompiling packages...
   2.0 s  ✓ LowLevelFEM → LowLevelFEMIterativeSolversExt
  1 dependency successfully precompiled in 4 seconds. 137 already precompiled.
[ Info: Precompiling LowLevelFEMPreconditionersExt [731aa7ef-afda-5ca4-bfd0-d21a0237399b] (caches not reused: 1 for different dependency version already loaded, 4 for different Julia build configuration)
Precompiling packages...
   2.6 s  ✓ LowLevelFEM → LowLevelFEMPreconditionersExt
  1 depen

In [2]:
openGeometry("I-beam.geo")

In [3]:
openPreProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()


In [4]:
mat = Material("beam")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)

Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 42589, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing))

In [5]:
per = MultiPointConstraint(master="right", slave="left", field=U)

MultiPointConstraint("right", "left", Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 42589, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), Dict{Symbol, Bool}())

In [6]:
e1 = VectorField(U, "left", [-1, 0, 0])
e2 = VectorField(U, "left", [0, -1, 0])
cs = CoordinateSystem(e1, e2, handedness=:left)

CoordinateSystem(VectorField([[-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;]  …  [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;], [-1.0; 0.0; … ; 0.0; 0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [1850, 1851, 1852, 1853, 1854, 1855, 1856, 1857, 1858, 1859  …  2188, 2189, 2190, 2191, 2192, 2193, 2194, 2195, 2196, 2197], 1, :v3D, Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 42589, LowLeve

In [7]:
K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="beam")
K[:,:]

127767×127767 SparseArrays.SparseMatrixCSC{Float64, Int64} with 8490739 stored entries:
⎡⣿⣿⣟⣛⣻⣛⣛⠛⠛⡯⠭⢿⠾⢿⣗⣒⣛⣒⣺⣬⣭⣟⣛⣛⣛⣛⣛⣛⠯⠿⠽⠾⢾⣓⣒⣗⣒⣧⣿⣬⎤
⎢⣿⢹⡻⣮⣸⠁⠈⠉⠉⡉⠉⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⣴⣞⠋⠉⢉⠉⠉⡉⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢺⠖⠚⠻⣦⡐⠀⠀⡇⠀⠐⠒⠒⠂⠀⠀⠀⠀⠀⠀⠒⠛⠛⢀⡼⠀⠀⠁⠂⠐⠒⠒⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠘⡆⠀⠐⠈⠻⣦⡀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠐⠋⠰⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⡿⡤⡇⠠⠤⠤⠤⠬⠿⣧⡀⢠⠀⠠⠤⠤⠤⠤⠤⠀⠀⠀⠀⠠⠤⠤⠬⠿⢦⣀⠀⠀⠠⠤⠤⠤⠤⠄⠀⠀⎥
⎢⣧⣇⡇⠀⢀⠀⠀⠀⠀⣈⣻⣾⡀⠀⡀⠀⠀⠀⢀⣀⣀⡀⠀⠀⠀⠀⠀⠀⣤⣼⣛⠀⣀⠀⠀⠀⠀⣀⣀⣀⎥
⎢⣾⣇⠀⠀⢸⠀⠀⠀⠀⡀⠀⠈⠻⣦⡇⠀⠀⠀⢀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠰⣿⣁⠀⠀⡀⠀⡀⠀⠀⎥
⎢⢹⢹⠀⠀⠈⠀⠀⠀⠀⡇⠀⠈⠉⠉⠻⣦⡉⠀⢸⠀⠀⠁⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠉⣡⡾⠃⠀⠀⠉⠀⎥
⎢⢻⢸⠀⠀⠀⠀⠀⠀⠀⡇⠀⠀⠀⠀⠃⠈⠻⣦⣸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠋⠀⠲⣄⠀⠀⠀⎥
⎢⡚⣾⠀⠀⠀⠀⠀⠀⠀⠃⠀⢰⠀⠐⠒⠒⠒⠚⠻⣦⡀⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠒⠒⠒⠚⠳⢤⣀⎥
⎢⣧⢿⢀⣤⢠⠀⠀⠀⠀⠀⠀⠸⠀⠀⠄⠀⠀⠀⠠⠬⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠴⠾⠯⎥
⎢⣿⢸⡾⠙⣿⠀⢀⠀⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⢀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢸⡇⢀⣀⡴⢋⡀⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢈⠻⣦⡙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢸⡇⠀⠀⠀⠈⠻⣦⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣌⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣯⡇⡇⠈⠡⠀⠀⠀⠈⢳⣀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣳⡇⠁⠀⢰⠀⠀⠀⠀⠀⠛⠘⣴⣦⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢾⢳⠀⠀⠘⠀⠀⠀⠀⡆⠀⠘⠁⠘⠇⣠⡶⠀⢠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⢦⡀⠀⠀⠀⎥
⎢⢼⢼⠀⠀⠀⠀⠀⠀⠀⡇⠀⠀⠀⠠⠾⠋⢠⡀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⣌⠻⣦⡙⠆⠀⠀⎥
⎢⠼⣼⠀⠀⠀⠀⠀⠀⠀⠇⠀⢠⠀⠠⠀⠀⠀⠙⢾⡀⢀⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⠌⠻⣦⡀⠀⎥
⎣⡛⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⠀⠀⠃⠀⠀⠀⠀⢳⡾⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [8]:
k = 1e4
KcL = ∫(U ⋅ k ⋅ U, Γ="left")
KcR = ∫(U ⋅ k ⋅ U, Γ="right")
(KcL+KcR)[:,:]

127767×127767 SparseArrays.SparseMatrixCSC{Float64, Int64} with 20940 stored entries:
⎡⡅⣭⡁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡅⠀⠀⠀⢨⠀⠀⡁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠁⠈⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠁⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠂⠒⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠐⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠄⠠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [9]:
f = ∫(U ⋅ [0,0,0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [10]:
ux = ScalarField(U, "right", (x,y,z)->y/10)
uy = ScalarField(U, "right", (x,y,z)->-x/10)
u0 = VectorField([ux, uy, 0])

elementwise VectorField
[[9.0; 5.0; … ; 5.0; 0.0;;], [9.333333333333332; 5.0; … ; 5.0; 0.0;;], [9.666666666666668; 5.0; … ; 5.0; 0.0;;], [9.0; 4.678571428571429; … ; 4.678187403993856; 0.0;;], [9.33333333333333; 4.678187403993856; … ; 4.677803379416281; 0.0;;], [9.666666666666663; 4.677803379416281; … ; 4.67741935483871; 0.0;;], [9.0; 4.357142857142857; … ; 4.35637480798771; 0.0;;], [9.333333333333332; 4.35637480798771; … ; 4.355606758832565; 0.0;;], [9.666666666666663; 4.355606758832565; … ; 4.354838709677419; 0.0;;], [9.0; 4.0357142857142865; … ; 4.0345622119815685; 0.0;;]  …  [-9.333333333333336; -3.712749615975421; … ; -3.7142857142857144; 0.0;;], [-10.0; -4.032258064516128; … ; -4.033410138248846; 0.0;;], [-9.666666666666668; -4.033410138248846; … ; -4.034562211981566; 0.0;;], [-9.333333333333336; -4.034562211981566; … ; -4.0357142857142865; 0.0;;], [-10.0; -4.354838709677419; … ; -4.355606758832565; 0.0;;], [-9.666666666666668; -4.355606758832565; … ; -4.3563748079877085; 0.0;;],

In [11]:
u = solveField(Symmetric(K + KcL + KcR), f - KcR * u0, mpc=[per], coordSys=[cs], solver=:cholesky);

In [12]:
showDoFResults(u, name="u", factor=1)

0

In [13]:
u = nodesToElements(u, onPhysicalGroup="beam")
ε = (u ∘ ∇ + ∇ ∘ u) / 2

E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1 - 2ν) * trace(ε) * I);

In [14]:
n = normalVector(U, "mid")

σ = elementsToElements(σ, fromPhysicalGroup="leftbeam", onPhysicalGroup="mid")

τ = σ * n
τ -= (τ ⋅ n) * n

τ = elementsToNodes(τ)
τ = nodesToElements(τ, onPhysicalGroup="mid")

elementwise VectorField
[[93.97063930763926; -132.44725748637117; … ; -186.36206082802306; 0.0;;], [-14.620861497694406; -186.36206082802306; … ; -186.8440402010301; 0.0;;], [-71.66649354842929; -186.8440402010301; … ; -145.52461817045364; 0.0;;], [102.69848322593245; -34.303406656251354; … ; -61.958974895476956; 0.0;;], [-43.41511457459825; -61.958974895476956; … ; -61.90999205285128; 0.0;;], [-149.02818084231538; -61.90999205285128; … ; -45.29442202311445; 0.0;;], [91.7827813575627; -9.043842352102311; … ; -21.105508993177363; 0.0;;], [-100.55665142936975; -21.105508993177363; … ; -21.097202436361375; 0.0;;], [-274.55188348369353; -21.097202436361375; … ; -19.069101539592204; 0.0;;], [31.590151202539417; -0.5122568087588755; … ; -7.763412310989745; 0.0;;]  …  [246.85367767699427; 3.4422708783161067; … ; -1.849745962231303; 0.0;;], [579.869094520708; 9.53490465127124; … ; 7.762468808285295; 0.0;;], [372.3997415721499; 7.762468808285295; … ; 7.7634123109754665; 0.0;;], [172.54807240033

In [15]:
r = nodePositionVector(U)
r = nodesToElements(r, onPhysicalGroup="mid")
∫(U, "mid", (r × τ) ⋅ n)

1.0424307983210395e8

In [16]:
showElementResults(τ, name="shear stress");

In [17]:
u1 = rotateResult(u, toCoordSys=[cs])

elementwise VectorField
[[-4.394402204464193; -2.4539031624948313; … ; 2.4770604761974524; -0.10736530066858523;;], [4.353840212198214; 2.4458022255866627; … ; 2.4652524415738477; -0.19869661586210388;;], [4.311799354344841; 2.4380336175995274; … ; 2.4471676414357266; -0.2812510784662258;;], [4.268745850946772; 2.422113092837093; … ; 2.423089229242364; -0.3588664388969376;;], [4.224273767816375; 2.3995301897282046; … ; 2.3940639010755866; -0.4330941101436707;;], [4.177335250039285; 2.3715472093072845; … ; 2.360842376378729; -0.5046298782505526;;], [4.127376209925827; 2.339174073287749; … ; 2.3240009549906824; -0.5738190169128076;;], [4.0740866045426385; 2.3030771045006677; … ; 2.28399896128789; -0.6408612959183118;;], [4.017298879243456; 2.263758398613976; … ; 2.241222109843244; -0.7058826469853116;;], [3.956924082516106; 2.2216243953555703; … ; 2.1959987110747097; -0.7689650989467325;;]  …  [4.057286290049839; 2.053924500164278; … ; 2.077760194244737; -0.6831786723431753;;], [4.125057

In [18]:
showElementResults(u1, name="local u")

2

In [19]:
openPostProcessor()